# Python `__new__()` 和 `__init__()`：继承与单例

本篇继续学习两个和对象创建有关的特殊方法：

- `__new__(cls, ...)`：创建并返回实例，发生在初始化之前。
- `__init__(self, ...)`：初始化已经创建好的实例，不负责创建实例。
- `__init__(self, ...)`：是 Python 中相当于面向对象编程 (OOP) 中构造函数的函数，它会在创建新对象时自动调用。会为对象的属性赋值，但不负责内存分配。
- 内存分配由 __new__() 方法处理，该方法在 __init__() 之前调用。

调用 `Child(...)` 时，可以先记住这条主线：

```text
调用类
  -> __new__ 创建对象
  -> 如果返回的对象是当前类实例，再调用 __init__
  -> 得到最终对象
```

## 1. 继承中的执行顺序

先看一个普通的父子类。子类没有重写 `__new__()` 时，会沿着继承关系找到父类的 `__new__()`；子类没有重写 `__init__()` 时，也会继承父类的 `__init__()`。

In [1]:
class Parent:
    def __new__(cls, *args, **kwargs):
        print(f"1. Parent.__new__: cls={cls.__name__}")
        obj = super().__new__(cls)
        print(f"2. Parent.__new__: obj={id(obj)}")
        return obj

    def __init__(self, name):
        print(f"3. Parent.__init__: self={id(self)}")
        self.name = name


class Child(Parent):
    pass

child = Child("小明")
print(f"4. child.name={child.name}")

1. Parent.__new__: cls=Child
2. Parent.__new__: obj=4378602320
3. Parent.__init__: self=4378602320
4. child.name=小明


输出顺序说明：先执行 `__new__()`，拿到对象后才执行 `__init__()`。注意，`__init__()` 接收到的 `self`，就是 `__new__()` 返回的那个对象。

如果父子类都重写了两个方法，子类方法需要显式调用 `super()`，调用链才会继续进入父类：

In [ ]:
class Base:
    def __new__(cls, *args, **kwargs):
        print("Base.__new__")
        return super().__new__(cls)

    def __init__(self, value):
        print("Base.__init__")
        self.value = value


class Derived(Base):
    def __new__(cls, *args, **kwargs):
        print("Derived.__new__")
        return super().__new__(cls, *args, **kwargs)

    def __init__(self, value):
        print("Derived.__init__")
        super().__init__(value)


item = Derived(42)
print(item.value)

### 继承时的关键规则

1. `Derived(...)` 首先查找并调用 `Derived.__new__()`。
2. 子类的 `__new__()` 通常通过 `super().__new__(cls)` 让父类继续创建对象。
3. 只有当 `__new__()` 返回的对象是 `Derived` 的实例时，Python 才会继续调用 `Derived.__init__()`。
4. `__init__()` 不会因为子类定义了自己的版本而自动调用父类版本；需要显式写 `super().__init__(...)`。
5. 如果子类重写 `__new__()` 却忘记返回对象，实例化通常会失败，因为后续没有可初始化的实例。

## 2. 为什么 `__init__()` 不能返回值？

`__init__()` 的职责是“就地初始化”已有对象，因此它必须返回 `None`。实例化表达式的结果已经由 `__new__()` 决定，Python 不允许 `__init__()` 再替换这个结果。

下面的代码会抛出 `TypeError`：

In [ ]:
class BadInit:
    def __init__(self):
        return "我想把实例替换成字符串"


try:
    BadInit()
except TypeError as error:
    print(type(error).__name__)
    print(error)

如果确实需要返回另一个对象，应该在 `__new__()` 中返回。此时，如果返回的对象不是当前类的实例，Python 不会调用当前类的 `__init__()`：

In [ ]:
class ReturnsString:
    def __new__(cls, text):
        return text

    def __init__(self, text):
        print("这行不会执行")


result = ReturnsString("hello")
print(result, type(result).__name__)

## 3. 用 `__new__()` 实现单例

单例的目标是：一个类无论实例化多少次，都只返回同一个对象。最直接的做法是在 `__new__()` 中缓存第一次创建的实例。

注意：下面的 `__init__()` 每次调用 `Singleton(...)` 仍然会执行，只是 `self` 每次都指向同一个对象。因此如果初始化成本很高或不能重复执行，需要额外加初始化标记。

In [ ]:
class Singleton:
    _instance = None

    def __new__(cls, *args, **kwargs):
        if cls._instance is None:
            print("第一次创建实例")
            cls._instance = super().__new__(cls)
        else:
            print("复用已有实例")
        return cls._instance

    def __init__(self, name):
        print(f"执行 __init__: name={name}")
        self.name = name


first = Singleton("第一次传入的名字")
second = Singleton("第二次传入的名字")

print(first is second)
print(first.name)

可以看到，`first is second` 为 `True`，但第二次调用仍然重新执行了 `__init__()`，所以 `name` 被改成了第二次传入的值。更稳妥的单例写法通常会让初始化只执行一次：

In [ ]:
class Config:
    _instance = None
    _initialized = False

    def __new__(cls):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
        return cls._instance

    def __init__(self):
        if self._initialized:
            return
        self.environment = "development"
        self._initialized = True


config_a = Config()
config_b = Config()
print(config_a is config_b)
print(config_b.environment)

### 单例实现的注意点

- 多线程环境下，`if cls._instance is None` 可能需要加锁，否则两个线程可能同时创建实例。
- 继承单例类时，实例缓存放在 `cls` 还是基类上，会影响子类是否各自拥有单例；需要根据语义明确设计。
- 很多场景使用模块级对象、依赖注入或显式传递配置会更简单；单例会引入全局状态，测试时要特别小心。
- `__new__()` 不只用于单例，也常用于创建不可变类型（例如 `str`、`tuple`）的子类，或根据参数决定返回哪一种对象。

## 一句话总结

`__new__()` 决定“要不要创建、创建哪个对象”，`__init__()` 决定“对象创建后如何填充状态”；继承时先走 `__new__()` 再走 `__init__()`，父类初始化必须用 `super()` 显式衔接，而单例正是利用 `__new__()` 把后续实例化请求导向同一个对象。